# Remote Sensing Image Segmentation with SatMAE++, SAM, and Wavelet-Based Strategies

**Authors**  
- Simone La Bella — Student ID: 1995847  
- Mattia Di Marco — Student ID: 2019367

## 1. Project Aim and Selected Papers

### Project Aim

The aim of this project is to investigate modern deep-learning strategies for **remote sensing image segmentation**, with particular focus on **building footprint segmentation** from aerial imagery.

The project uses the **INRIA Aerial Image Labeling Dataset** as the main downstream dataset and studies how different techniques can improve segmentation quality under both fully supervised and label-scarce conditions.

The main objectives are:

- evaluate the effectiveness of a **SatMAE++ pretrained Vision Transformer** as an encoder for building segmentation;
- compare the SatMAE++ based model with a **Swin-Unet baseline**;
- investigate whether **SAM-generated pseudo-labels** can reduce the amount of manually annotated data required for training;
- explore **wavelet-based strategies** for preserving spatial details and high-frequency information that may be useful for accurately segmenting building boundaries;
- perform controlled comparisons between the different approaches using common segmentation metrics such as **IoU, Dice score, precision, recall, and accuracy**.

### Selected Papers

The project is mainly inspired by the following papers.

#### 1. *Rethinking Transformers Pre-training for Multi-Spectral Satellite Imagery*

This work introduces **SatMAE++**, a self-supervised pretraining strategy designed specifically for remote sensing imagery. The method extends masked autoencoder pretraining to both RGB and multi-spectral satellite data and learns representations that can be transferred to downstream remote sensing tasks.

In this project, SatMAE++ provides the main pretrained encoder. The pretrained ViT-Large model is adapted from its original pretraining setting to a **dense prediction task**, namely binary building footprint segmentation on the INRIA dataset.

#### 2. *SAMRS: Scaling-up Remote Sensing Segmentation Dataset with Segment Anything Model*

SAMRS explores the use of the **Segment Anything Model (SAM)** to automatically generate segmentation masks for remote sensing imagery starting from spatial prompts such as bounding boxes.

Inspired by this approach, our project uses a **SatMAE segmentation teacher** to produce coarse building predictions. Connected components are converted into bounding-box prompts, which are then passed to SAM to generate refined pseudo-labels.

To avoid data leakage, the teacher is trained only on a small labeled subset of the INRIA training set. The remaining training images are treated as unlabeled data and are pseudo-labeled using the SatMAE teacher and SAM.

#### 3. *A Wavelet Diffusion GAN for Image Super-Resolution*

This paper investigates the integration of **wavelet transforms** into a generative super-resolution framework. Its main idea is to process image and feature representations in the wavelet domain so that low-frequency structural information and high-frequency details can be handled more effectively.

Instead of reproducing the diffusion-GAN architecture, we adapt the frequency-aware principle to the segmentation decoder.

#### 4. *WaveDM: Wavelet-Based Diffusion Models for Image Restoration*

WaveDM proposes a wavelet-based diffusion framework for image restoration. One of its central ideas is the separate treatment of **low-frequency and high-frequency components** in the wavelet domain.

This idea is adapted in our segmentation pipeline through a **wavelet-aware decoder**. Intermediate decoder features are decomposed using a Haar Discrete Wavelet Transform into low-frequency and high-frequency components.

The components are processed through separate branches, reconstructed using an inverse wavelet transform, and combined with the original features through a residual connection.


## 2. Theoretical Background and Key Concepts

### Semantic Segmentation

Semantic segmentation is a dense prediction task in which a class label is assigned to every pixel of an input image.

In this project, the problem is formulated as **binary semantic segmentation**:

- **building**;
- **background**.

Given an input aerial image $I$, the model produces a probability map

$$
P \in [0,1]^{H \times W}
$$

where each value represents the probability that the corresponding pixel belongs to a building.

A threshold, typically $0.5$, is then applied to obtain the final binary segmentation mask.

Building footprint segmentation is particularly challenging because the model must not only identify the presence of buildings, but also accurately reconstruct their **shape and boundaries**. Small buildings, adjacent structures, shadows, vegetation, and differences in illumination can make this task difficult.

---

### Vision Transformers for Remote Sensing

Vision Transformers (ViTs) process an image as a sequence of image patches rather than relying exclusively on local convolutional operations.

An image is divided into fixed-size patches, which are projected into embedding vectors and processed using Transformer blocks based on self-attention.

This allows the network to model long-range relationships between different regions of an image.

For remote sensing imagery, this property can be particularly useful because relevant objects and structures may cover large spatial regions and their interpretation can depend on the surrounding context.

In our project, the main encoder is based on a **ViT-Large architecture pretrained using SatMAE++**.

---

### Self-Supervised Pretraining and SatMAE++

Training large Vision Transformers from scratch generally requires large amounts of annotated data.

Self-supervised learning reduces this dependency by learning useful visual representations from unlabeled images.

SatMAE++ follows the idea of **Masked Autoencoders (MAE)**. During pretraining, a large fraction of the input image patches is masked and the model is trained to reconstruct the missing information from the visible patches.

Conceptually:

```text
Satellite image
      ↓
Split into patches
      ↓
Random masking
      ↓
Visible patches → Transformer encoder
      ↓
Reconstruction of masked information
```

Through this task, the encoder learns general representations of remote sensing imagery without requiring dense semantic labels.

The pretrained encoder can then be transferred to a downstream task through **fine-tuning**.

---

### Encoder-Decoder Segmentation Architecture

The SatMAE++ encoder produces a compact representation of the input image.

With an input resolution of $224 \times 224$ and patches of size $16 \times 16$, the spatial representation corresponds to a grid of:

$$
\frac{224}{16} \times \frac{224}{16}
=
14 \times 14
$$

The Transformer therefore produces high-level features at a significantly lower spatial resolution than the original image.

A segmentation decoder progressively increases the feature-map resolution:

```text
14 × 14
   ↓
28 × 28
   ↓
56 × 56
   ↓
112 × 112
   ↓
224 × 224
   ↓
Binary segmentation mask
```

The encoder is mainly responsible for extracting semantic information, while the decoder reconstructs the spatial resolution required for pixel-level prediction.

---

### Transfer Learning and Fine-Tuning

The pretrained SatMAE++ weights provide an initialization that already contains representations learned from remote sensing imagery.

During fine-tuning, both the encoder and segmentation decoder are optimized for the INRIA building segmentation task.

Different learning rates are used for the two components:

```text
Pretrained encoder → lower learning rate
New decoder        → higher learning rate
```

This allows the decoder to adapt quickly to the new task while modifying the pretrained representation more conservatively.

The project also allows comparison between pretrained and non-pretrained configurations to evaluate the actual contribution of SatMAE++ pretraining.

---

### Pseudo-Labeling and Semi-Supervised Learning

Dense segmentation masks are expensive to produce manually.

**Pseudo-labeling** is a semi-supervised learning technique in which a model trained on labeled data is used to generate artificial labels for unlabeled samples.

The newly labeled samples can then be added to the training set.

In our project, the INRIA training data is divided into:

```text
10% labeled data
90% treated as unlabeled data
```

A SatMAE++ teacher model is trained exclusively on the 10% labeled subset.

The remaining 90% is then processed by the teacher to obtain coarse building predictions.

This separation is important because the teacher must never use the ground-truth masks of the samples that will later receive pseudo-labels. Otherwise, the experiment would suffer from **data leakage**.

The leak-free pipeline is therefore:

```text
10% ground truth
      ↓
SatMAE teacher
      ↓
Predictions on unseen 90%
      ↓
Pseudo-label generation
      ↓
10% real labels + 90% pseudo-labels
      ↓
Final student model
```

---

### Segment Anything Model and Prompt-Based Segmentation

The **Segment Anything Model (SAM)** is a promptable segmentation model.

Instead of directly predicting a semantic class such as "building", SAM receives spatial prompts, for example:

- points;
- bounding boxes;
- masks.

and returns candidate segmentation masks corresponding to the specified region.

Because SAM is class-agnostic, it cannot directly identify which regions correspond to buildings.

For this reason, SatMAE is first used to produce a coarse semantic prediction.

Connected components are extracted from this prediction and converted into bounding boxes:

```text
INRIA image
     ↓
SatMAE teacher
     ↓
Coarse building mask
     ↓
Connected components
     ↓
Bounding-box prompts
     ↓
SAM
     ↓
Refined pseudo-label
```

SAM therefore acts as a **mask refinement mechanism**, while SatMAE provides the semantic information required to identify candidate building regions.

---

### Discrete Wavelet Transform

The Discrete Wavelet Transform (DWT) decomposes a signal into components representing different spatial-frequency information.

For a two-dimensional feature map, a one-level Haar DWT produces four components:

$$
LL,\quad LH,\quad HL,\quad HH
$$

where:

- **LL** represents the low-frequency component and mainly preserves coarse structures and global information;
- **LH**, **HL**, and **HH** represent high-frequency components associated with local variations, edges, and fine details.

Conceptually:

```text
Feature map
     ↓
    DWT
     ↓
┌────┬────┬────┬────┐
│ LL │ LH │ HL │ HH │
└────┴────┴────┴────┘
```

This decomposition is particularly relevant for building segmentation.

Low-frequency information can help represent the general shape and spatial extent of a building, while high-frequency information can contain useful information about building boundaries, corners, roof edges, separation between adjacent structures, and small spatial details.

---

### Wavelet-Aware Feature Processing

Inspired by wavelet-based image restoration and super-resolution approaches, the project introduces wavelet processing directly inside the segmentation decoder.

Instead of treating all feature information identically, intermediate feature maps are decomposed using the Haar DWT.

The low-frequency component and high-frequency components are then processed using separate branches:

```text
                     ┌→ LL → Low-Frequency Branch ─────┐
Feature map → DWT ───┤                                 │
                     └→ LH / HL / HH → High-Freq Branch
                                                       │
                                                       ↓
                                                      IDWT
                                                       ↓
                                              Reconstructed features
                                                       ↓
                                             Residual connection
```

The reconstructed wavelet features are combined with the original feature map through a residual connection.

This allows the network to preserve the original representation while learning additional frequency-aware corrections.

In the current architecture, wavelet-aware blocks are introduced at intermediate decoder resolutions:

```text
56 × 56
112 × 112
```

rather than at the initial $14 \times 14$ representation, where the spatial resolution is already highly compressed.

---

### Residual Learning

Residual connections combine the original representation $F$ with a learned transformation $G(F)$:

$$
F_{\mathrm{out}} = F + G(F)
$$

In the proposed wavelet-aware decoder, $G(F)$ corresponds to the features obtained after DWT, frequency-specific processing, and inverse DWT.

The residual formulation allows the model to retain the original decoder features while adding frequency-domain information only when it is useful.

---

### Training Objective

The main segmentation models are trained using a combination of **Binary Cross-Entropy (BCE)** and **Dice Loss**.

Binary Cross-Entropy evaluates pixel-wise classification:

$$
\mathcal{L}_{BCE}
=
-\frac{1}{N}
\sum_{i=1}^{N}
\left[
y_i \log(p_i)
+
(1-y_i)\log(1-p_i)
\right]
$$

while Dice Loss directly encourages overlap between the predicted and ground-truth masks:

$$
\mathcal{L}_{Dice}
=
1 -
\frac{
2\sum_{i=1}^{N} p_i y_i + \epsilon
}{
\sum_{i=1}^{N} p_i +
\sum_{i=1}^{N} y_i +
\epsilon
}
$$

The combined objective used in the main experiments is:

$$
\mathcal{L}_{total}
=
0.5\,\mathcal{L}_{BCE}
+
0.5\,\mathcal{L}_{Dice}
$$

Using both objectives combines pixel-level supervision with an overlap-based objective particularly suitable for segmentation problems.

## 3. Implementation Details

### Dataset

The experiments are conducted using the **INRIA Aerial Image Labeling Dataset**, a remote sensing dataset designed for building footprint segmentation.

Each image is associated with a binary segmentation mask in which building pixels are labeled as foreground and all remaining pixels are considered background.

The dataset is therefore used as a binary segmentation problem:

- **1 — Building**
- **0 — Background**

The official INRIA test set does not provide public ground-truth masks. For this reason, the experiments use the annotated training portion of the dataset and create an internal train/validation split.

To reduce the risk of spatial data leakage, the split is performed at the **source-image level before patch extraction**. This ensures that patches extracted from the same original aerial image cannot appear in both the training and validation sets.

In the current configuration, images from **Vienna** are used as the validation subset, while the remaining annotated source images are used for training.

After preprocessing, the dataset contains approximately:

| Split | Number of patches |
|---|---:|
| Training | 14,400 |
| Validation | 3,600 |

The processed dataset is organized as:

```text
datasets/inria_processed/
├── train/
│   ├── images/
│   └── masks/
├── val/
│   ├── images/
│   └── masks/
├── test/
│   └── images/
└── metadata.csv
```

The `metadata.csv` file stores information for every extracted patch, including:

```text
patch_id
split
city
source_image
top
left
tile_size
source_height
source_width
building_ratio
image_path
mask_path
```

This metadata is also used later to construct the semi-supervised experiments and to guarantee that the labeled and pseudo-labeled subsets are generated without source-image overlap.

---

### Dataset Preprocessing

The original aerial images are considerably larger than the input resolution required by the neural networks. A preprocessing pipeline is therefore applied before training.

#### Patch Extraction

Each original aerial image and its corresponding segmentation mask are divided into patches of:

$$
512 \times 512
$$

pixels.

The image and mask are cropped using exactly the same coordinates so that the pixel-level correspondence between them is preserved.

Each patch is stored independently and its original position is recorded in `metadata.csv` through the `top` and `left` coordinates.

The preprocessing can therefore be summarized as:

```text
Original aerial image
        +
Ground-truth mask
        ↓
Train/validation split by source image
        ↓
512 × 512 patch extraction
        ↓
Store image/mask pairs
        ↓
Generate metadata.csv
```

Importantly, the train/validation division is performed **before** patch extraction. If patches were first generated and randomly divided afterwards, neighboring regions from the same original image could appear in both sets and artificially increase validation performance.

---

### Input Resizing

Although the stored patches have resolution:

$$
512 \times 512
$$

the models operate on:

$$
224 \times 224
$$

inputs.

Images are therefore resized during data loading rather than permanently replacing the original processed patches.

For RGB images, bicubic interpolation is used:

```text
512 × 512 RGB patch
        ↓
Bicubic resize
        ↓
224 × 224 RGB input
```

Segmentation masks must preserve discrete labels, so they are resized using **nearest-neighbor interpolation**:

```text
512 × 512 binary mask
        ↓
Nearest-neighbor resize
        ↓
224 × 224 binary mask
```

Using nearest-neighbor interpolation prevents the creation of artificial intermediate label values along building boundaries.

After resizing, the mask is binarized to ensure that each pixel belongs either to the building or background class.

---

### SatMAE++ Input Preprocessing

The SatMAE++ checkpoint used in this project was pretrained on the **FMoW-RGB** dataset.

To remain consistent with the preprocessing used by the pretrained model, the RGB tensor is converted to **BGR channel order** before normalization.

The normalization statistics used are:

```python
mean = (
    0.4182007312774658,
    0.4214799106121063,
    0.3991275727748871
)

std = (
    0.28774282336235046,
    0.27541765570640564,
    0.2764017581939697
)
```

Therefore, the complete SatMAE preprocessing pipeline is:

```text
512 × 512 RGB patch
        ↓
Resize to 224 × 224
        ↓
Convert tensor from RGB to BGR
        ↓
Normalize with FMoW-RGB statistics
        ↓
SatMAE++ encoder
```

This preprocessing is shared by the SatMAE baseline, pseudo-label models, Wavelet Loss experiment, and Wavelet-Aware Decoder experiment.

---

### Swin-Unet Input Preprocessing

The Swin-Unet baseline uses the original Swin-Unet implementation and its pretrained ImageNet weights.

Images are resized to:

$$
224 \times 224
$$

and normalized using standard ImageNet statistics:

```python
mean = (0.485, 0.456, 0.406)
std  = (0.229, 0.224, 0.225)
```

The segmentation masks follow the same nearest-neighbor resizing procedure used for SatMAE.

---

### Data Augmentation

Data augmentation is applied only to the training set.

The transformations used include:

- horizontal flipping;
- vertical flipping;
- rotations by multiples of $90^\circ$;
- mild brightness variation;
- mild contrast variation;
- mild saturation variation;
- mild hue variation.

Spatial transformations are applied identically to both the input image and its segmentation mask.

Color transformations are applied only to the RGB image.

No augmentation is applied to the validation set.

The objective is to expose the models to variations in orientation and appearance while preserving the semantic structure of buildings.

---

### SatMAE++ Segmentation Baseline

The main model is based on a **SatMAE++ ViT-Large encoder** initialized using the pretrained checkpoint:

```text
checkpoint_ViT-L_pretrain_fmow_rgb.pth
```

For an input image of:

$$
224 \times 224
$$

with patch size:

$$
16 \times 16
$$

the encoder produces a spatial representation of:

$$
14 \times 14
$$

with an embedding dimension of 1024.

A convolutional decoder progressively reconstructs the original spatial resolution:

```text
SatMAE++ ViT-L Encoder
        ↓
14 × 14 × 1024
        ↓
Convolutional projection
        ↓
28 × 28
        ↓
56 × 56
        ↓
112 × 112
        ↓
224 × 224
        ↓
1-channel segmentation logits
```

The entire network is fine-tuned end-to-end.

A lower learning rate is applied to the pretrained encoder and a higher learning rate to the newly initialized decoder.

---

### Swin-Unet Baseline

As a second baseline, the project uses the original **Swin-Unet** architecture.

The model is initialized using:

```text
swin_tiny_patch4_window7_224.pth
```

and configured for binary segmentation by setting the number of output classes to one.

The Swin-Unet baseline provides a transformer-based segmentation reference against which the SatMAE-based approaches can be compared.

---

### SatMAE++ with Wavelet Loss

A first wavelet-based experiment preserves the standard SatMAE segmentation architecture but modifies the training objective.

The predicted probability mask and the ground-truth mask are decomposed using a one-level **Haar Discrete Wavelet Transform**.

The resulting high-frequency components:

```text
LH
HL
HH
```

are compared using an additional L1-based wavelet loss.

The training objective becomes conceptually:

$$
\mathcal{L}_{total}
=
\mathcal{L}_{BCE+Dice}
+
\lambda\mathcal{L}_{wavelet}
$$

with the initial configuration using:

$$
\lambda = 0.1
$$

This experiment evaluates whether directly encouraging consistency in high-frequency mask components improves the reconstruction of building boundaries.

---

### SatMAE++ with Wavelet-Aware Decoder

A second wavelet strategy modifies the decoder rather than the loss function.

The SatMAE++ pretrained encoder remains unchanged.

Wavelet-Aware Feature Blocks are inserted into the decoder at spatial resolutions:

```text
56 × 56
112 × 112
```

For each feature map, a one-level Haar DWT produces:

```text
LL
LH
HL
HH
```

The low-frequency component `LL` is processed using a dedicated low-frequency branch, while `LH`, `HL`, and `HH` are processed by a separate high-frequency branch.

The processed components are then reconstructed using an inverse wavelet transform:

```text
Feature map
     ↓
Haar DWT
     ↓
┌──────────────┬──────────────────┐
│ LL           │ LH / HL / HH     │
│              │                  │
│ Low-Freq     │ High-Freq        │
│ Branch       │ Branch           │
└──────┬───────┴─────────┬────────┘
       └────────┬────────┘
                ↓
              IDWT
                ↓
      Reconstructed features
                ↓
       Residual connection
```

A residual connection combines the reconstructed frequency-aware representation with the original decoder features.

For this experiment, the loss remains the standard BCE + Dice objective so that the effect of the architectural modification can be evaluated independently from the Wavelet Loss experiment.

---

### SAM Pseudo-Label Generation

The project also investigates whether SAM-generated pseudo-labels can reduce the dependency on manually annotated segmentation masks.

To create a controlled label-scarce experiment, the original INRIA training split is further divided at the **source-image level** into:

```text
10% labeled data
90% treated as unlabeled data
```

A SatMAE++ teacher is initialized from the original FMoW-RGB pretrained checkpoint and fine-tuned **only on the 10% labeled subset**.

This restriction is essential to prevent data leakage.

The trained teacher predicts coarse masks on the remaining 90%:

```text
Unlabeled INRIA patch
        ↓
SatMAE teacher
        ↓
Coarse building probability mask
        ↓
Thresholding
        ↓
Connected components
```

Connected components smaller than a minimum area are removed.

For each remaining component, a bounding box is generated and slightly expanded.

The bounding boxes are then provided to SAM as prompts:

```text
Coarse SatMAE mask
        ↓
Connected components
        ↓
Bounding boxes
        ↓
SAM
        ↓
Candidate masks
        ↓
Best matching mask
        ↓
Pseudo-label
```

When SAM returns multiple candidate masks, they are evaluated using both the SAM confidence score and their overlap with the original SatMAE component.

The resulting masks are stored under:

```text
datasets/inria_processed/pseudo_leakfree/
```

A new metadata file is generated:

```text
metadata_semisupervised_leakfree.csv
```

The final student model is trained using:

```text
10% real ground-truth masks
+
90% SAM pseudo-labels
```

The student model is initialized again from the **original SatMAE++ pretrained checkpoint**, rather than from the teacher.

This prevents information from the teacher training process from being implicitly transferred to the final model.

---

### Experimental Setup

All main experiments use comparable optimization settings whenever possible.

| Parameter | Value |
|---|---:|
| Input resolution | $224 \times 224$ |
| Maximum epochs | 20 |
| Early stopping patience | 5 |
| Physical batch size | 1 |
| Gradient accumulation | 4 |
| Effective batch size | 4 |
| Encoder learning rate | $1 \times 10^{-5}$ |
| Decoder learning rate | $1 \times 10^{-4}$ |
| Weight decay | 0.05 |
| Optimizer | AdamW |
| Main loss | BCE + Dice |
| BCE weight | 0.5 |
| Dice weight | 0.5 |
| Prediction threshold | 0.5 |

Automatic Mixed Precision (AMP) is enabled during training when CUDA is available.

For SatMAE-based models, the encoder and decoder use separate learning rates because the encoder is already pretrained while the segmentation decoder is trained for the new downstream task.

A cosine learning-rate schedule is applied during training.

The model with the highest validation IoU is stored as the best checkpoint.

---

### Evaluation Metrics

Model performance is evaluated using the following metrics:

- **Intersection over Union (IoU)**;
- **Dice coefficient**;
- **Precision**;
- **Recall**;
- **Pixel Accuracy**.

IoU is used as the primary metric for model selection and early stopping.

All quantitative comparisons are performed on the same validation subset and using the same decision threshold of $0.5$ to ensure that the models are evaluated under consistent conditions.

## 4. Results and Analysis

The different models were evaluated using the same validation set and the same segmentation metrics. Training behavior is first analyzed through the evolution of the loss and validation metrics. The final models are then compared quantitatively using IoU, Dice, Precision, Recall, and Accuracy.

---

### 4.1 Swin-Unet Baseline

The following plots show the training and validation loss together with the evolution of the segmentation metrics during training.

<p align="center">
  <img src="outputs/swinunet_baseline/plots/loss.png" width="48%">
  <img src="outputs/swinunet_baseline/plots/metrics.png" width="48%">
</p>

---

### 4.2 SatMAE++ Baseline

The SatMAE++ baseline uses the pretrained FMoW-RGB ViT-Large encoder followed by the segmentation decoder described in the previous section.

<p align="center">
  <img src="outputs/satmae_baseline/plots/loss.png" width="48%">
  <img src="outputs/satmae_baseline/plots/metrics.png" width="48%">
</p>

---

### 4.3 SatMAE++ with Wavelet Loss

In this experiment, the standard SatMAE++ architecture is trained using the additional high-frequency Haar Wavelet Loss.

<p align="center">
  <img src="outputs/satmae_wavelet/plots/loss.png" width="48%">
  <img src="outputs/satmae_wavelet/plots/metrics.png" width="48%">
</p>

---

### 4.4 SatMAE++ with Wavelet-Aware Decoder

This experiment modifies the segmentation decoder by introducing Wavelet-Aware Feature Blocks at intermediate spatial resolutions.

<p align="center">
  <img src="outputs/satmae_wavelet_decoder/plots/loss.png" width="48%">
  <img src="outputs/satmae_wavelet_decoder/plots/metrics.png" width="48%">
</p>

---

### 4.5 SatMAE++ with SAM Pseudo-Labels

The semi-supervised experiment uses 10% of the INRIA training set with real ground-truth annotations and the remaining 90% with leak-free pseudo-labels generated through the SatMAE teacher and SAM pipeline.

<p align="center">
  <img src="outputs/satmae_sam_pseudolabels_leakfree/plots/loss.png" width="48%">
  <img src="outputs/satmae_sam_pseudolabels_leakfree/plots/metrics.png" width="48%">
</p>

---

### 4.6 Quantitative Evaluation

All models are evaluated on the same validation set using a prediction threshold of $0.5$.

| Model | IoU | Dice | Precision | Recall | Accuracy |
|---|---:|---:|---:|---:|---:|
| Swin-Unet | - | - | - | - | - |
| SatMAE++ Baseline | - | - | - | - | - |
| SatMAE++ + Wavelet Loss | - | - | - | - | - |
| SatMAE++ + Wavelet-Aware Decoder | - | - | - | - | - |
| SatMAE++ + SAM Pseudo-Labels | - | - | - | - | - |

The values in this table are obtained using the unified evaluation pipeline described previously. IoU is considered the primary metric, while Dice, Precision, Recall, and Accuracy provide complementary information about segmentation performance.

---

### 4.7 Qualitative Evaluation

In addition to the quantitative evaluation, the predictions of the different models are visually inspected on the same validation sample.

Each visualization contains the original aerial image, the ground-truth segmentation mask, the predicted mask, and an overlay of the prediction on the original image.

#### Swin-Unet

<p align="center">
  <img src="outputs/evaluation/swin/single/PATCH_ID/comparison.png" width="90%">
</p>

#### SatMAE++ Baseline

<p align="center">
  <img src="outputs/evaluation/satmae/single/PATCH_ID/comparison.png" width="90%">
</p>

#### SatMAE++ with Wavelet Loss

<p align="center">
  <img src="outputs/evaluation/satmae_wavelet/single/PATCH_ID/comparison.png" width="90%">
</p>

#### SatMAE++ with Wavelet-Aware Decoder

<p align="center">
  <img src="outputs/evaluation/satmae_wavelet_decoder/single/PATCH_ID/comparison.png" width="90%">
</p>

#### SatMAE++ with SAM Pseudo-Labels

<p align="center">
  <img src="outputs/evaluation/satmae_sam/single/PATCH_ID/comparison.png" width="90%">
</p>

Using the same validation image for all models makes it possible to qualitatively compare differences in building localization, boundary reconstruction, false positives, and missed structures.